# 01 — Exploração da curva de juros

Pré-requisito: `make pipeline` (gera `data/gold/yield_curve.parquet`). A lógica de produção fica em `src/`; este notebook apenas explora e reporta.

In [ ]:
import sys
from pathlib import Path

import plotly.express as px
import polars as pl

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

RANDOM_SEED = 42
curve = pl.read_parquet(ROOT / "data" / "gold" / "yield_curve.parquet")
curve.head()

## Cobertura temporal e número de vencimentos por título

In [ ]:
curve.group_by("bond_type").agg(
    pl.col("reference_date").min().alias("first_date"),
    pl.col("reference_date").max().alias("last_date"),
    pl.col("maturity_date").n_unique().alias("n_maturities"),
)

## Taxa de um prazo constante (~3 anos) ao longo do tempo

Para cada data, escolhe-se o vencimento mais próximo de 3 anos.

In [ ]:
three_years = (
    curve.with_columns((pl.col("years_to_maturity") - 3).abs().alias("distance"))
    .sort("distance")
    .group_by(["bond_type", "reference_date"])
    .first()
    .filter(pl.col("distance") < 0.5)
    .sort("reference_date")
)
fig = px.line(
    three_years,
    x="reference_date",
    y="buy_rate",
    color="bond_type",
    labels={"reference_date": "Data", "buy_rate": "Taxa de compra (% a.a.)", "bond_type": "Título"},
    title="Taxa de compra com prazo próximo de 3 anos",
)
fig